In [ ]:
import re
import os
import sys
import json
import pickle
import glob
import numpy as np

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import CenteredNorm, TwoSlopeNorm
import seaborn as sns

fontsize = 9
lw = 0.75
matplotlib.rc('font', **{'family': 'Arial', 'size': fontsize})
matplotlib.rc('axes', **{'linewidth': 0.75, 'labelsize': fontsize})
matplotlib.rc('xtick', **{'labelsize': fontsize})
matplotlib.rc('ytick', **{'labelsize': fontsize})
matplotlib.rc('xtick.major', **{'width': lw, 'size':3})
matplotlib.rc('ytick.major', **{'width': lw, 'size':3})
matplotlib.rc('ytick.minor', **{'width': lw, 'size':1.5})

In [ ]:
data_dir = os.path.join('..','sims')
data_files = glob.glob(os.path.join(data_dir, 'Efield_*_*.npz'))

values = []
for f in data_files:
    match = re.findall('\d+', os.path.basename(f))
    values.append((int(match[0]), float(match[1])))
dtype = [('index',int), ('amp',float)]
A = np.array(values, dtype=dtype)
idx = np.argsort(A, order=['index','amp'])
A = A[idx]
stim_index = np.unique(A['index']).tolist()
stim_amp = np.unique(A['amp'])

data_files = [data_files[i] for i in idx]
N_data_files = len(data_files)
print(f'Found {N_data_files} data files.')

In [ ]:
data = [np.load(f, allow_pickle=True) for f in data_files]
time_to_next_spike = {amp: np.zeros(len(stim_index)) for amp in stim_amp}
for a,D in zip(A,data):
    stim_time = D['config'].item()['stim']['Efield']['delay']
    idx = np.where(D['spike_times'] > stim_time)[0][0]
    jdx = stim_index.index(a[0])
    time_to_next_spike[a[1]][jdx] = D['spike_times'][idx] - stim_time

In [ ]:
spont_data_file = os.path.join('..','sims','Efield_0.npz')
spont_data = np.load(spont_data_file, allow_pickle=True)

t,V = spont_data['time'],spont_data['Vsoma']
spike_times = spont_data['spike_times']
# ISI = np.diff(spike_times*1e-3)
# F = 1 / ISI

stim_time = data[0]['config'].item()['stim']['Efield']['delay']
idx = np.where(spike_times < stim_time)[0][-1]
t0,t1 = spike_times[idx],spike_times[idx+1]
stim_times = np.linspace(t0+0.5, t1-0.5, len(stim_index))
assert stim_time == stim_times[0]

T = t1 - t0
# spike_phase = spike_time / T
period_idx = (t>=t0) & (t<=t1)

In [ ]:
normalized = False
cmap = plt.get_cmap('plasma', len(time_to_next_spike))
col = 'tab:green'
fig,ax = plt.subplots(1, 1, figsize=(6,4))
twin_ax = ax.twinx()
spike_time = t[period_idx] - t0
if normalized:
    spike_time /= T
twin_ax.plot(spike_time, V[period_idx], lw=2, color=col)

spike_time = stim_times - t0
if normalized:
    spike_time /= T
for i,(amp,dlys) in enumerate(time_to_next_spike.items()):
    y = dlys
    y[y>5] = np.nan
    ax.plot(spike_time, y, 'o-', color=cmap(i), markerfacecolor='w', markersize=4,
            lw=1, markeredgewidth=1, label=f'E={amp:.0f} V/m')
ax.legend(loc='upper left', bbox_to_anchor=(0.1,0.4,0.2,0.6), frameon=True, fontsize=7)
if normalized:
    ax.set_xlabel('Spike phase')
    ax.set_xticks(np.r_[0 : 1.1 : 0.1])
else:
    ax.set_xlabel('Time in spike (ms)')
ax.set_ylabel('Time to next spike (ms)')
limits = np.array([[0,2.5], [-75,50]])
ticks = [np.r_[limits[0,0] : limits[0,1] : 0.3], np.r_[limits[1,0] : limits[1,1] : 15]]
ax.set_yticks(ticks[0])
twin_ax.set_yticks(ticks[1])
ax.set_ylim(limits[0])
twin_ax.set_ylim(limits[1])
twin_ax.set_ylabel('Vm (mV)', color=col)
twin_ax.tick_params(axis='y', labelcolor=col)
ax.grid(which='major', axis='y', ls=':', lw=0.5, color=[.6,.6,.6])
fig.tight_layout()
plt.savefig('time_to_spike' + ('_normalized.pdf' if normalized else '.pdf'))